# Love Laundry Chatbot

A conversational bot for Love Laundry — handles customer queries about services, pricing, pickup/delivery, and general FAQs.

**Status:** Skeleton / untrained. Plug in training data and retrain to improve responses.

In [ ]:
import json
import re
import random
from typing import Optional

## 1. Intent Definitions

Each intent has example phrases and a set of possible responses. Add more intents and training phrases here to improve the bot.

In [ ]:
INTENTS = {
    "greeting": {
        "patterns": [
            "hello", "hi", "hey", "good morning", "good evening",
            "what's up", "sup", "howdy", "hi there"
        ],
        "responses": [
            "Hello! Welcome to Love Laundry. How can I help you today?",
            "Hey there! Need help with laundry? I'm here for you.",
            "Hi! Thanks for reaching out to Love Laundry. What can I do for you?"
        ]
    },
    "services": {
        "patterns": [
            "what services", "what do you offer", "services list",
            "what can you do", "laundry services", "what do you provide",
            "wash and fold", "ironing", "dry cleaning", "pickup delivery"
        ],
        "responses": [
            "We offer 4 main services:\n\n"
            "1. **Wash & Fold** — Everyday laundry, washed and folded\n"
            "2. **Ironing** — Crisp, pressed clothes\n"
            "3. **Dry Cleaning** — Special care for suits, dresses, delicates\n"
            "4. **Pickup & Delivery** — We collect and return to your door\n\n"
            "Would you like to know more about any of these?"
        ]
    },
    "pricing": {
        "patterns": [
            "how much", "price", "pricing", "cost", "rates",
            "what does it cost", "charge", "fees", "tariff"
        ],
        "responses": [
            "Our pricing depends on the service and volume. Here's a general idea:\n\n"
            "• Wash & Fold: From Rs. 200/kg\n"
            "• Ironing: From Rs. 50/piece\n"
"
            "• Dry Cleaning: From Rs. 300/piece\n\n"
            "For an exact quote, call us at +94 70 000 0000 or WhatsApp us!"
        ]
    },
    "pickup_delivery": {
        "patterns": [
            "pickup", "delivery", "collect", "drop off",
            "when do you pick up", "delivery time", "how long",
            "schedule pickup", "book pickup", "arrange collection"
        ],
        "responses": [
            "We offer free pickup and delivery! Here's how it works:\n\n"
            "1. Call or WhatsApp us to schedule\n"
            "2. We collect your laundry at the arranged time\n"
            "3. We wash, fold/iron, and deliver back fresh!\n\n"
            "Typical turnaround is 24–48 hours. Same-day service available on request."
        ]
    },
    "locations": {
        "patterns": [
            "where are you", "location", "address", "where to drop",
            "find you", "nearest branch", "collection points",
            "madampe", "mahawewa", "chilaw", "whennappuwa"
        ],
        "responses": [
            "Our main centre is in Chilaw, Puttalam District.\n\n"
            "We also have collection points in:\n"
            "• Madampe\n• Mahawewa\n• Kottaramulla\n• Dunakadeniya\n• Bibiladeniya\n• Wennappuwa\n\n"
            "Scroll down on our website to see all locations on the map!"
        ]
    },
    "hours": {
        "patterns": [
            "opening hours", "when are you open", "working hours",
            "business hours", "time", "schedule", "open now"
        ],
        "responses": [
            "We're available 24/7 for WhatsApp bookings!\n\n"
            "Pickup and delivery hours:\n"
            "• Monday – Saturday: 8:00 AM – 7:00 PM\n"
            "• Sunday: 9:00 AM – 5:00 PM\n\n"
            "Call us anytime at +94 70 000 0000."
        ]
    },
    "contact": {
        "patterns": [
            "contact", "phone", "whatsapp", "call you",
            "get in touch", "reach you", "number"
        ],
        "responses": [
            "You can reach us through:\n\n"
            "📞 Phone: +94 70 000 0000\n"
            "💬 WhatsApp: +94 70 000 0000\n"
            "📧 Email: info@lovelaundry.lk\n\n"
            "We'd love to hear from you!"
        ]
    },
    "commercial": {
        "patterns": [
            "hotel", "business", "commercial", "bulk",
            "restaurant", "spa", "gym", "hotel linen",
            "corporate", "big order"
        ],
        "responses": [
            "Yes! We serve hotels, restaurants, spas, and businesses.\n\n"
            "Our commercial services include:\n"
            "• Hotel Linen Service — bed sheets, towels, table linens\n"
            "• Commercial Laundry — restaurant linens, spa towels, uniforms\n"
            "• Bulk Processing — daily/weekly pickup, fast turnaround\n\n"
            "We currently partner with 9 hotels including Goldi Sands, Amagi, and Camelot.\n"
            "Contact us for a custom quote!"
        ]
    },
    "quality": {
        "patterns": [
            "quality", "how good", "careful", "safety",
            "will you damage", "stain", "delicate", "silk"
        ],
        "responses": [
            "We treat every garment with professional care!\n\n"
            "• Professional-grade equipment\n"
            "• Separate handling for delicates\n"
            "• Quality inspection before delivery\n"
            "• 4.9/5 customer rating\n\n"
            "Your clothes are in safe hands with Love Laundry."
        ]
    },
    "careers": {
        "patterns": [
            "job", "hiring", "work", "career", "join team",
            "employment", "vacancy", "apply"
        ],
        "responses": [
            "We're always looking for great people to join our team!\n\n"
            "Current openings:\n"
            "• Delivery Driver\n"
            "• Machine Operator\n"
            "• Ironer / Presser\n"
            "• Collection Agent\n\n"
            "Benefits include competitive pay, training, and flexible schedules.\n"
            "Send us a WhatsApp message to apply!"
        ]
    },
    "thanks": {
        "patterns": [
            "thank", "thanks", "thank you", "cheers",
            "appreciate", "helpful"
        ],
        "responses": [
            "You're welcome! Is there anything else I can help with?",
            "Happy to help! Let me know if you need anything else.",
            "My pleasure! Feel free to ask anytime."
        ]
    },
    "goodbye": {
        "patterns": [
            "bye", "goodbye", "see you", "talk later",
            "that's all", "nothing else", "done"
        ],
        "responses": [
            "Goodbye! Have a great day. Your clothes deserve Love Laundry!",
            "See you! Don't forget to book your next pickup.",
            "Bye! We're here whenever you need us."
        ]
    }
}

## 2. Intent Classifier

Simple keyword-matching classifier. Replace with an ML model (e.g. scikit-learn, spaCy, or OpenAI) for better accuracy.

In [ ]:
def preprocess(text: str) -> str:
    """Lowercase, strip punctuation, collapse whitespace."""
    text = text.lower().strip()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text


def classify_intent(user_message: str) -> Optional[str]:
    """Match user message against known intent patterns."""
    cleaned = preprocess(user_message)
    best_intent = None
    best_score = 0

    for intent, data in INTENTS.items():
        score = 0
        for pattern in data["patterns"]:
            if pattern in cleaned:
                # Longer pattern matches are worth more
                score += len(pattern.split())
        if score > best_score:
            best_score = score
            best_intent = intent

    return best_intent if best_score > 0 else None

## 3. Response Generator

In [ ]:
FALLBACK_RESPONSE = (
    "I'm not sure I understand that. Could you rephrase?\n\n"
    "I can help with:\n"
    "• Our services and pricing\n"
    "• Pickup & delivery scheduling\n"
    "• Locations and collection points\n"
    "• Commercial / hotel laundry\n"
    "• Job openings\n\n"
    "Or call us at +94 70 000 0000 for direct help."
)


def get_response(user_message: str) -> str:
    """Classify intent and return a response."""
    intent = classify_intent(user_message)

    if intent is None:
        return FALLBACK_RESPONSE

    return random.choice(INTENTS[intent]["responses"])

## 4. Test the Bot

In [ ]:
test_messages = [
    "Hi there!",
    "What services do you offer?",
    "How much does ironing cost?",
    "Can you pick up from Mahawewa?",
    "I run a hotel, can you handle our linen?",
    "What time do you open?",
    "I want to apply for a job",
    "random nonsense xyz 123",
    "Thank you!",
]

for msg in test_messages:
    intent = classify_intent(msg)
    response = get_response(msg)
    print(f"User: {msg}")
    print(f"Intent: {intent}")
    print(f"Bot: {response[:80]}...")
    print("-" * 60)

## 5. API Endpoint (Future)

When ready to deploy, uncomment and run this cell to start a Flask/FastAPI server.

```python
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route("/chat", methods=["POST"])
def chat():
    data = request.get_json()
    message = data.get("message", "")
    response = get_response(message)
    return jsonify({"response": response})

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000, debug=True)
```

### Future Improvements
- Train with real customer conversation data
- Add NLP model (spaCy, transformers, or OpenAI API)
- Store conversation history
- Add sentiment analysis
- Connect to booking system
- Multi-language support (Sinhala, Tamil)